#### __libraries__

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

In [11]:
from shared_scripts.statistical_analysis import t_margin_of_error, permutation_test_median, iqr_outlier_removal, two_sample_t_test

In [12]:
from shared_scripts.bb2_data_load import BB2_load_into_dataframe

In [13]:
from shared_scripts.additional_functions import extract_significance_symbols

#### __plt params__

In [15]:
plt.rcParams["legend.frameon"] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.labelsize'] = 19
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16  
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['font.family'] = 'Calibri'

#### __func__

In [17]:
def _8C_data_process(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
        
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in data.dtypes):
        raise TypeError("All columns in inout must be numerical")
        
    #assign timepoint
    frq= kwargs.get('frequency', np.NaN)
    indel= kwargs.get('initial_delay', np.NaN)
    if "frequency" not in kwargs or "initial_delay" not in kwargs:
        raise ValueError("frequency and initial_delay must be provided")
    try:
        data= data.assign(timepoint_minutes=data['frame'] * frq + indel).drop(columns=["frame"])
    except Exception as ex:
        raise ValueError("failed to calculate the timepoints in minutes")
        
    #set the index 
    if 'timepoint_minutes' in data.columns:
        data= data.set_index('timepoint_minutes')
    else:
        raise ValueError("dataset is missing a frame column")
        
    #calculate the mean
    try:
        data= data.assign(avg_distance_ctrl= data.iloc[:, 0:3].mean(axis= 1),
                          avg_distance_brfa= data.iloc[:, 3:6].mean(axis= 1))
    except Exception as ex:
        print(f'failed to calculate the mean: {ex}')
        data= data.assign(avg_distance_ctrl= np.NaN,
                          avg_distance_brfa= np.NaN)
        
    #calculate the MOE95
    try:
        data= data.assign(moe95_ctrl= data.apply(lambda x: t_margin_of_error(x.iloc[0:3]), axis= 1),
                          moe95_brfa= data.apply(lambda x: t_margin_of_error(x.iloc[3:6]), axis= 1))
    except Exception as ex:
        print(f'failed to calculate the moe95: {ex}')
        data= data.assign(moe95_ctrl= np.NaN,
                          moe95_brfa= np.NaN)
        
    return data


def _8D_F_data_processing(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
    
    #chck the columns
    required_columns = {'group', 'metal_concentration', 'timepoint_minutes', 'percentage_of_cells_with_foci', 'avg_number_of_foci_per_cell', 'avg_size_single_focus'}
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        raise ValueError(f"Input dataset is missing required columns: {sorted(missing_columns)}")
        
    #group biological repeats into lists
    try:
        data= data.groupby(['group', 'metal_concentration', 'timepoint_minutes']).agg({'percentage_of_cells_with_foci':list,
                                                                                       'avg_number_of_foci_per_cell':list,
                                                                                       'avg_size_single_focus':list}).reset_index()
    except Exception as ex:
        raise ValueError(f'Failed to convert the biological repeats into lists: {ex}')
        
        
    #calculate the descriptive stats
    try:
        data= data.assign(percentage_of_cells_with_foci_mean= data.percentage_of_cells_with_foci.apply(lambda x: np.array(x).mean()),
                          percentage_of_cells_with_foci_std= data.percentage_of_cells_with_foci.apply(lambda x: np.array(x).std(ddof= 1)),
                          percentage_of_cells_with_foci_moe95= data.percentage_of_cells_with_foci.apply(lambda x: t_margin_of_error(x)),
                          avg_number_of_foci_per_cell_mean= data.avg_number_of_foci_per_cell.apply(lambda x: np.array(x).mean()),
                          avg_number_of_foci_per_cell_std= data.avg_number_of_foci_per_cell.apply(lambda x: np.array(x).std(ddof= 1)),
                          avg_number_of_foci_per_cell_moe95= data.avg_number_of_foci_per_cell.apply(lambda x: t_margin_of_error(x)),
                          avg_size_single_focus_mean= data.avg_size_single_focus.apply(lambda x: np.array(x).mean()),
                          avg_size_single_focus_std= data.avg_size_single_focus.apply(lambda x: np.array(x).std(ddof=1)),
                          avg_size_single_focus_moe95= data.avg_size_single_focus.apply(lambda x: t_margin_of_error(x)))
    except Exception as ex:
        raise ValueError(f'Failed to calculate the descriptive statistics: {ex}')
        
    #add a corresponding control data (for statistical testing)
    control_data= data.loc[data.group=='solvent control', ['metal_concentration', 'timepoint_minutes', 'percentage_of_cells_with_foci', 'avg_number_of_foci_per_cell', 'avg_size_single_focus']]
    control_data.columns= ['metal_concentration','timepoint_minutes', 'percentage_of_cells_with_foci_ctrl', 'avg_number_of_foci_per_cell_ctrl', 'avg_size_single_focus_ctrl']
    data= data.merge(control_data, how= 'left', on= ['metal_concentration', 'timepoint_minutes'])
    
    #t test and significance symbols
    p_values_thr= kwargs.get('p_value_threshold', 0.05)
    try:
        data= data.assign(percentage_of_cells_with_foci_pvalue= data.apply(lambda x: two_sample_t_test(x['percentage_of_cells_with_foci'], x['percentage_of_cells_with_foci_ctrl']), axis= 1),
                          avg_number_of_foci_per_cell_pvalue= data.apply(lambda x: two_sample_t_test(x['avg_number_of_foci_per_cell'], x['avg_number_of_foci_per_cell_ctrl']), axis= 1),
                          avg_size_single_focus_pvalue= data.apply(lambda x: two_sample_t_test(x['avg_size_single_focus'], x['avg_size_single_focus_ctrl']), axis= 1))
        data= data.assign(percentage_of_cells_with_foci_significance= np.where(data.percentage_of_cells_with_foci_pvalue==1, '', np.where(data.percentage_of_cells_with_foci_pvalue<p_values_thr, '*', '')),
                          avg_number_of_foci_per_cell_significance= np.where(data.avg_number_of_foci_per_cell_pvalue==1, '', np.where(data.avg_number_of_foci_per_cell_pvalue<p_values_thr, '*', '')),
                          avg_size_single_focus_significance= np.where(data.avg_size_single_focus_pvalue==1, '', np.where(data.avg_size_single_focus_pvalue<p_values_thr, '*', '')))
    except Exception as ex:
        raise ValueError(f'Failed to calculate p-values and assign a significance symbols: {ex}')
           
    #reduce to needed columns
    data= data.loc[:, ['group', 'metal_concentration', 'timepoint_minutes', 
                       'percentage_of_cells_with_foci_mean', 'percentage_of_cells_with_foci_std', 'percentage_of_cells_with_foci_moe95', 'percentage_of_cells_with_foci_pvalue', 'percentage_of_cells_with_foci_significance',
                       'avg_number_of_foci_per_cell_mean', 'avg_number_of_foci_per_cell_std', 'avg_number_of_foci_per_cell_moe95', 'avg_number_of_foci_per_cell_pvalue', 'avg_number_of_foci_per_cell_significance',
                       'avg_size_single_focus_mean', 'avg_size_single_focus_std', 'avg_size_single_focus_moe95', 'avg_size_single_focus_pvalue', 'avg_size_single_focus_significance']]
    
    #assign stages
    data= data.assign(stage= pd.cut(data.timepoint_minutes,
                                    bins= [0, 139.33333333333334, 200.66666666666666, 1000], #stage bins calculation (elsewhere) for 20240502
                                    labels= ['Formation', 'Coalescence/Fusion', 'Clearance']))
    
    #sort groups: control-solvent control-inhibitor (assign numerical index)
    data= data.assign(group_index= (np.where(data.group=='solvent control', 1, 2)))
    data= data.sort_values(['group_index', 'metal_concentration', 'timepoint_minutes'], ignore_index= True)  
        
        
    return data


def _8E_data_process(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
    
    #chck the columns
    required_columns = {'timepoint_minutes', 'inhibitor_abbreviation', 'number_of_foci', 'fov_cell_id'}
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        raise ValueError(f"Input dataset is missing required columns: {sorted(missing_columns)}")
        
    #filter down to a selected timepoint
    selected_timepoint= kwargs.get('selected_timepoint', 60)
    if not isinstance(selected_timepoint, (int, float)) or selected_timepoint not in data.timepoint_minutes.unique():
        raise ValueError("Selected timepoint must be numeric and in the dataset's timepoint_minutes column")
    data= data.loc[data.timepoint_minutes == selected_timepoint]
    
    #assign the label
    data= data.assign(label= np.where((data.metal_concentration > 0) & (data.inhibitor_abbreviation == '-'), f'As(III)', f'As(III) + BrfaA'))
    
    #filter to non-zero and selected-max agg counts
    max_agg_count= kwargs.get('max_no_of_aggregates', 6)
    if not isinstance(max_agg_count, (int, float)):
        raise ValueError("Selected max agg-count must be numeric")
    data= data.loc[(data.number_of_foci > 0) & (data.number_of_foci <= max_agg_count)]
    
    #pivot
    data= data.pivot_table(index= 'label', columns= 'number_of_foci', values= 'fov_cell_id', aggfunc= 'size')
    
    data= data.apply(lambda x: round(x/x.sum(), 2), axis= 1)
    
    return data

#### __data load__

In [19]:
_8B= BB2_load_into_dataframe('Figure_8B.csv')

In [20]:
_8C= BB2_load_into_dataframe('Figure_8C.csv')

In [21]:
_8D_8F= BB2_load_into_dataframe('Figure_8D_8F.csv')

In [22]:
_8E= BB2_load_into_dataframe('Figure_8E.csv')

#### __data processing__

In [24]:
_8C_final= _8C_data_process(_8C, frequency= 6, initial_delay= 6)

In [25]:
_8D_8F_final= _8D_F_data_processing(_8D_8F)

In [54]:
_8E_final_120min= _8E_data_process(_8E, selected_timepoint= 120, max_no_of_aggregates= 6)
_8E_final_198min= _8E_data_process(_8E, selected_timepoint= 198, max_no_of_aggregates= 6)

#### __permutation test__

In [28]:
# permutation_test_median(_8B, category_column= 'group',  reference_group= 'met_ctrl', tested_parameter= 'alpha', n_resamples= 100000, export= True, path= r"permutation_test\Fig_8B_alpha_permutation_test.csv")

In [29]:
# permutation_test_median(_8B, category_column= 'group', reference_group= 'met_ctrl', tested_parameter= 'K', n_resamples= 100000, export= True, path= r"permutation_test\Fig_8B_K_permutation_test.csv")

#### __significance symbols__

In [31]:
significance_alpha= extract_significance_symbols(path= r"permutation_test\Fig_8B_alpha_permutation_test.csv")
significance_k= extract_significance_symbols(path= r"permutation_test\Fig_8B_K_permutation_test.csv")

#### __visuals__

In [61]:
def fig6(_8B_data, _8C_data, _8D_8F_data, _8E_data_120, _8E_data_198, **kwargs):
    
    #fig + gridspec
    fig= plt.figure(figsize= (12.5, 20), constrained_layout= True)
    gs= gridspec.GridSpec(8, 6, figure=fig)

    ax1= fig.add_subplot(gs[0:2, 0:3])
    ax2= fig.add_subplot(gs[0:2, 3:6])
    ax3= fig.add_subplot(gs[2:4, 0:6])
    ax4= fig.add_subplot(gs[4:6, 0:3])
    ax5= fig.add_subplot(gs[4:6, 3:6])
    ax6= fig.add_subplot(gs[6:7, 0:3])
    ax7= fig.add_subplot(gs[6:7, 3:6])
    ax8= fig.add_subplot(gs[7:8, 0:2])
    ax9= fig.add_subplot(gs[7:8, 2:4])
    ax10= fig.add_subplot(gs[7:8, 4:6])
    
    ###fig 8B
    #alpha
    box_position = 1.5 
    points_bottom, points_top= 0.25, 0.75 
    colors= plt.cm.tab10(np.linspace(0, 1, 10))
    for group in _8B_data.group.unique():
        dark= '#00527C' if group == 'met_ctrl' else '#FF781F'
        light= '#DFE9F5' if group == 'met_ctrl' else '#FEE7CC'
        data= _8B_data.loc[_8B_data.group== group]
        ax1.boxplot(data.alpha, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_alpha= kwargs.get('significance_symbols_alpha', None)
        if symbols_alpha:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.alpha))
            ax1.annotate(symbols_alpha.get(group, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center', fontsize= 16)
        x_points = np.random.uniform(points_bottom, points_top, len(data.alpha))
        ax1.scatter(x_points,
                      data.alpha,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        #K
        ax2.boxplot(data.K, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_k= kwargs.get('significance_symbols_k', None)
        if symbols_k:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.K))
            ax2.annotate(symbols_k.get(group, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center', fontsize= 16)
        ax2.scatter(x_points,
                      data.K,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        box_position += 2
        points_bottom += 2
        points_top += 2
    #rename the groups
    _8B_data= _8B_data.assign(group= np.where(_8B_data.group=='met_ctrl', 'As(III)', 'As(III) + BrfA'))
    ax1.set_ylabel('alpha', weight= 'bold')
    ax1.set_ylim(bottom=-0.55, top= 2.55)
    ax1.axhline(_8B_data.loc[_8B_data.group=='As(III)', 'alpha'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax1.set_xlim(0, len(_8B_data.group.unique())*2)
    ax1.set_xticks(np.arange(1, len(_8B_data.group.unique())*2, 2))
    ax1.set_xticklabels(_8B_data.group.unique())
    ax1.set_yticks([-0.5, 0.5, 1.5, 2.5])
    ax2.set_ylabel('K', weight= 'bold')
    ax2.set_ylim(bottom=-2, top= 122)
    ax2.axhline(_8B_data.loc[_8B_data.group=='As(III)', 'K'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax2.set_xlim(0, len(_8B_data.group.unique())*2)
    ax2.set_xticks(np.arange(1, len(_8B_data.group.unique())*2, 2))
    ax2.set_xticklabels(_8B_data.group.unique())
    ax2.set_yticks([0, 40, 80, 120])
    
    ###fig 8C
    ax3.errorbar(_8C_data.index, _8C_data.avg_distance_ctrl, lw= 2.5, label= 'As(III)', color= '#00527C')
    y_min_ctrl= _8C_data.avg_distance_ctrl - _8C_data.moe95_ctrl
    y_max_ctrl= _8C_data.avg_distance_ctrl + _8C_data.moe95_ctrl
    ax3.fill_between(_8C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#DFE9F5',  label="margin of error area: As(III)", alpha= .66)
    ax3.errorbar(_8C_data.index, _8C_data.avg_distance_brfa, lw= 2.5, label= 'As(III) + BrfA', color= '#FF781F')
    y_min_ctrl= _8C_data.avg_distance_brfa - _8C_data.moe95_brfa
    y_max_ctrl= _8C_data.avg_distance_brfa + _8C_data.moe95_brfa
    ax3.fill_between(_8C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#FEE7CC',  label="margin of error area: As(III) + BrfA", alpha= .66)
    ax3.set_xlabel('time (min)', weight= 'bold')
    ax3.set_ylabel('average movement', weight= 'bold') #pxls/frame
    ax3.set_ylim(0, 8)
    ax3.set_yticks([1, 3, 5, 7])
    ax3.legend(ncol= 2)
    ax3.set_xticks([0,60,120,180,240,300,360])
    
    ###fig 8D a
    solvent_control_data= _8D_8F_data.loc[(_8D_8F_data.group=='solvent control') & (_8D_8F_data.metal_concentration=='0.5 mM')]
    cc= round(solvent_control_data.loc[solvent_control_data.stage=='Formation', 'avg_number_of_foci_per_cell_mean'].corr(solvent_control_data.loc[solvent_control_data.stage=='Formation', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax4.scatter(solvent_control_data.loc[solvent_control_data.stage=='Formation', 'avg_number_of_foci_per_cell_mean'],
                solvent_control_data.loc[solvent_control_data.stage=='Formation', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#DFE9F5',
                label= f'FO, {cc}')
    x= solvent_control_data.loc[solvent_control_data.stage=='Formation', 'avg_number_of_foci_per_cell_mean']
    y= solvent_control_data.loc[solvent_control_data.stage=='Formation', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax4.plot(x, 
             m*x+b,
             color= '#00527C',
             linewidth= 2.5)
    
    cc= round(solvent_control_data.loc[solvent_control_data.stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'].corr(solvent_control_data.loc[solvent_control_data.stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax4.scatter(solvent_control_data.loc[solvent_control_data.stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'],
                solvent_control_data.loc[solvent_control_data.stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#FEE7CC',
                label= f'C/F, {cc}')
    x= solvent_control_data.loc[solvent_control_data.stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean']
    y= solvent_control_data.loc[solvent_control_data.stage=='Coalescence/Fusion', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax4.plot(x, 
             m*x+b,
             color= '#FF781F',
             linewidth= 2.5)
    
    cc= round(solvent_control_data.loc[solvent_control_data.stage=='Clearance', 'avg_number_of_foci_per_cell_mean'].corr(solvent_control_data.loc[solvent_control_data.stage=='Clearance', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax4.scatter(solvent_control_data.loc[solvent_control_data.stage=='Clearance', 'avg_number_of_foci_per_cell_mean'],
                solvent_control_data.loc[solvent_control_data.stage=='Clearance', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#D6FED2',
                label= f'CL, {cc}')
    x= solvent_control_data.loc[solvent_control_data.stage=='Clearance', 'avg_number_of_foci_per_cell_mean']
    y= solvent_control_data.loc[solvent_control_data.stage=='Clearance', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax4.plot(x, 
             m*x+b,
             color= '#0C8001',
             linewidth= 2.5)
    solvent_control_control_data= _8D_8F_data.loc[(_8D_8F_data.group=='solvent control') & (_8D_8F_data.metal_concentration=='0.0 mM')]
    ax4.scatter(solvent_control_control_data.loc[:, 'avg_number_of_foci_per_cell_mean'],
                solvent_control_control_data.loc[:, 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#ECECEC',
                label= f'control')
    ax4.set_xlabel('aggregates/cell', weight= 'bold')
    ax4.set_ylabel('aggregate size', weight= 'bold')
    ax4.legend(frameon= True, loc= 'upper right', ncol= 2)
    ax4.set_xlim(0, 3.5)
    ax4.set_ylim(0.12, 0.4)
    ax4.set_yticks([0.2, 0.3, 0.4, 0.5])
    ax4.text(00.70, 0.1, 
         f'As(III)',
         transform=ax4.transAxes,
         fontweight= 'bold', fontsize= 18,
         verticalalignment='center',
         horizontalalignment='center')
    ax4.set_xlim(0, 3)
    
    ###fig 8D b
    brfa_exposed_data= _8D_8F_data.loc[(_8D_8F_data.group=='BrfA-exposed') & (_8D_8F_data.metal_concentration=='0.5 mM')]
    cc= round(brfa_exposed_data.loc[brfa_exposed_data.stage=='Formation', 'avg_number_of_foci_per_cell_mean'].corr(brfa_exposed_data.loc[brfa_exposed_data.stage=='Formation', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax5.scatter(brfa_exposed_data.loc[brfa_exposed_data.stage=='Formation', 'avg_number_of_foci_per_cell_mean'],
                brfa_exposed_data.loc[brfa_exposed_data.stage=='Formation', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#DFE9F5',
                label= f'FO, {cc}')
    x= brfa_exposed_data.loc[brfa_exposed_data.stage=='Formation', 'avg_number_of_foci_per_cell_mean']
    y= brfa_exposed_data.loc[brfa_exposed_data.stage=='Formation', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax5.plot(x, 
             m*x+b,
             color= '#00527C',
             linewidth= 2.5)
    
    cc= round(brfa_exposed_data.loc[brfa_exposed_data.stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'].corr(brfa_exposed_data.loc[brfa_exposed_data.stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax5.scatter(brfa_exposed_data.loc[brfa_exposed_data.stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'],
                brfa_exposed_data.loc[brfa_exposed_data.stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#FEE7CC',
                label= f'C/F, {cc}')
    x= brfa_exposed_data.loc[brfa_exposed_data.stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean']
    y= brfa_exposed_data.loc[brfa_exposed_data.stage=='Coalescence/Fusion', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax5.plot(x, 
             m*x+b,
             color= '#FF781F',
             linewidth= 2.5)
    
    cc= round(brfa_exposed_data.loc[brfa_exposed_data.stage=='Clearance', 'avg_number_of_foci_per_cell_mean'].corr(brfa_exposed_data.loc[brfa_exposed_data.stage=='Clearance', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax5.scatter(brfa_exposed_data.loc[brfa_exposed_data.stage=='Clearance', 'avg_number_of_foci_per_cell_mean'],
                brfa_exposed_data.loc[brfa_exposed_data.stage=='Clearance', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#D6FED2',
                label= f'CL, {cc}')
    x= brfa_exposed_data.loc[brfa_exposed_data.stage=='Clearance', 'avg_number_of_foci_per_cell_mean']
    y= brfa_exposed_data.loc[brfa_exposed_data.stage=='Clearance', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax5.plot(x, 
             m*x+b,
             color= '#0C8001',
             linewidth= 2.5)
    brfa_exposed_control_data= _8D_8F_data.loc[(_8D_8F_data.group=='BrfA-exposed') & (_8D_8F_data.metal_concentration=='0.0 mM')]
    ax5.scatter(brfa_exposed_control_data.loc[:, 'avg_number_of_foci_per_cell_mean'],
                brfa_exposed_control_data.loc[:, 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#ECECEC',
                label= f'control')
    ax5.set_xlabel('aggregates/cell', weight= 'bold')
    ax5.set_ylabel('aggregate size', weight= 'bold')
    ax5.legend(frameon= True, loc= 'upper right', ncol= 2)
    ax5.set_xlim(0, 3.5)
    ax5.set_ylim(0.12, 0.4)
    ax5.set_yticks([0.2, 0.3, 0.4, 0.5])
    ax5.text(00.70, 0.1, 
         f'As(III) + BrfA',
         transform=ax5.transAxes,
         fontweight= 'bold', fontsize= 18,
         verticalalignment='center',
         horizontalalignment='center')
    ax5.set_xlim(0, 3)
    
    ###fig 8e
    sns.heatmap(_8E_data_120,
                cmap= 'coolwarm',                
                annot= True,
                fmt= '.2f',
                ax= ax6,
                linecolor= 'white',
                linewidths= 1,
                annot_kws={"size": 15},
                cbar= False)
    ax6.tick_params(axis='y', labelrotation=45)
    ax6.set_xlabel('aggregates/cell')
    ax6.set_ylabel('')
    for label in ax6.get_yticklabels():
        label.set_horizontalalignment('right')
        label.set_x(-0.01)
        
    sns.heatmap(_8E_data_198,
                cmap= 'coolwarm',                
                annot= True,
                fmt= '.2f',
                ax= ax7,
                linecolor= 'white',
                linewidths= 1,
                annot_kws={"size": 15},
                cbar= False)
    ax7.tick_params(axis='y', labelrotation=45)
    ax7.set_xlabel('aggregates/cell')
    ax7.set_ylabel('')
    for label in ax7.get_yticklabels():
        label.set_horizontalalignment('right')
        label.set_x(-0.01)
    
    
    ###fig 8f
    axes= [ax8, ax9, ax10]
    sel_tmpts=  kwargs.get('selected_timepoints', [60, 180, 300])
    if (not isinstance(sel_tmpts, list) or len(sel_tmpts) != 3 or not all(isinstance(i, (int, float)) for i in sel_tmpts)):
        raise ValueError("Input must be a list of exactly 3 numerical elements.")
    ctrl_patch = mpatches.Patch(color='#DFE9F5', label='control')
    exposed_patch = mpatches.Patch(color='#FEE7CC', label='As(III)')
    for i, tmpt in enumerate(sel_tmpts):
        x= np.arange(len(_8D_8F_data.group.unique()))
        width= 0.4
        control= _8D_8F_data.loc[(_8D_8F_data.timepoint_minutes==tmpt)&(_8D_8F_data.metal_concentration=='0.0 mM'), ['percentage_of_cells_with_foci_mean', 'percentage_of_cells_with_foci_moe95', 'percentage_of_cells_with_foci_significance']].reset_index(drop= True)
        as_exposed= _8D_8F_data.loc[(_8D_8F_data.timepoint_minutes==tmpt)&(_8D_8F_data.metal_concentration=='0.5 mM'), ['percentage_of_cells_with_foci_mean', 'percentage_of_cells_with_foci_moe95', 'percentage_of_cells_with_foci_significance']].reset_index(drop= True)
        axes[i].bar(x-width/2,
                    control.percentage_of_cells_with_foci_mean,
                    yerr= control.percentage_of_cells_with_foci_moe95,
                    color= '#DFE9F5',
                    edgecolor= '#00527C',
                    width=width,
                    linewidth= .66,
                    error_kw={'ecolor': 'black', 'elinewidth': .66, 'capsize': 2, 'capthick': .66, 'alpha': 0.8})
        x_coor= list(control.index - width/2)
        y_coor= list(control.percentage_of_cells_with_foci_mean + control.percentage_of_cells_with_foci_moe95 + 4)
        for s, (xc, yc) in enumerate(zip(x_coor, y_coor)):
            axes[i].annotate(control.percentage_of_cells_with_foci_significance[s], (xc, yc), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center', fontsize= 16)
        axes[i].bar(x+width/2,
                    as_exposed.percentage_of_cells_with_foci_mean,
                    yerr= as_exposed.percentage_of_cells_with_foci_moe95,
                    width=width,
                    color= '#FEE7CC',
                    edgecolor= '#FF781F',
                    linewidth= .66,
                    error_kw={'ecolor': 'black', 'elinewidth': .66, 'capsize': 2, 'capthick': .66, 'alpha': 0.8})
        x_coor= list(as_exposed.index + width/2)
        y_coor= list(as_exposed.percentage_of_cells_with_foci_mean + as_exposed.percentage_of_cells_with_foci_moe95 + 4)
        for s, (xc, yc) in enumerate(zip(x_coor, y_coor)):
            axes[i].annotate(as_exposed.percentage_of_cells_with_foci_significance[s], (xc, yc), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center', fontsize= 16)
        axes[i].legend(handles=[ctrl_patch, exposed_patch], loc= 'upper right', ncol= 2, frameon= False)
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(['control', 'BrfA'])
        axes[i].set_ylabel('cells with\naggregates (%)', weight= 'bold', fontsize= 15)
        axes[i].set_ylim(-10, 140)
        axes[i].set_yticks([0, 50, 100])

In [63]:
fig6(_8B, 
     _8C_final,
     _8D_8F_final,
     _8E_final_120min,
     _8E_final_198min,
     significance_symbols_alpha= significance_alpha,
     significance_symbols_k= significance_k,
     selected_timepoints= [90, 180, 360])